In [1]:
from jqdatasdk import *
import configparser
 
config = configparser.ConfigParser()
file = 'config.ini'
config.read(file)
user = config.get('login','user')
password = config.get('login','password')
auth(user,password)

auth success （JQData现有流量增加活动，详情请咨询JQData管理员，微信号：JQData02）


# 获取沪深指数数据
## 指数概况
包含指数的上市时间、退市时间、代码、名称等。平台支持的指数见指数列表。

### 获取单支指数数据
获取单支指数的信息.

**调用方法**

get_security_info(code)

**参数**

code: 指数代码

**返回值**

 一个对象, 有如下属性:

display_name # 中文名称  
name # 缩写简称  
start_date # 上市日期, [datetime.date] 类型  
end_date # 退市日期，[datetime.date] 类型, 如果没有退市则为2200-01-01  
type # 类型，index(指数)  

**示例**

In [4]:
# 输出沪深300指数开始时间
get_security_info('000300.XSHG').start_date

datetime.date(2005, 4, 8)

### 获取所有指数数据
获取平台支持的所有指数数据

**调用方法**

get_all_securities(types=['index'], date=None)  
这里请在使用时注意防止未来函数。

**返回**
 display_name # 中文名称  
 name # 缩写简称  
 start_date # 上市日期  
 end_date # 退市日期，如果没有退市则为2200-01-01  
 type # 类型，index(指数)  

[pandas.DataFrame], 比如:get_all_securities(['index'])[:2]返回:

In [5]:
get_all_securities(['index'])[:2]

,display_name,name,start_date,end_date,type
000001.XSHG,上证指数,SZZS,1991-07-15,2200-01-01,index
000002.XSHG,A股指数,AGZS,1992-02-21,2200-01-01,index


### 获取指数成份股
获取一个指数给定日期在平台可交易的成分股列表，我们支持近600种股票指数数据，包括指数的行情数据以及成分股数据。为了避免未来函数，我们支持获取历史任意时刻的指数成分股信息。请点击指数列表查看指数信息.

**调用方法**

get_index_stocks(index_symbol, date=None)

**参数**
- index_symbol, 指数代码
- date: 查询日期, 一个字符串(格式类似’2015-10-15’)或者[datetime.date]/[datetime.datetime]对象, 可以是None, 使用默认日期. 这个默认日期在回测和研究模块上有点差别:

1. 回测模块: 默认值会随着回测日期变化而变化, 等于context.current_dt
2. 研究模块: 默认是今天

**返回**
- 返回股票代码的list

**示例**

In [6]:
# 获取所有沪深300的股票, 设为股票池
indexs = get_index_stocks('000300.XSHG')
# set_universe(indexs)

### 获取指数成份股权重
获取给定日期和指数的指数成分股权重数据，指数数据请查看下面的沪深指数列表

**调用方法**

get_index_weights(index_id, date=None)

**参数**

index_id: 代表指数的标准形式代码， 形式：指数代码.交易所代码，例如"000001.XSHG"。  
date: 查询权重信息的日期，形式："%Y-%m-%d"，例如"2018-05-03"；date可以是None，当date=None时，返回最近一次更新的指数成份股权重。  

**返回**

查询到对应日期，且有权重数据，返回 pandas.DataFrame， code(股票代码)，display_name(股票名称), date(日期), weight(权重)；  
查询到对应日期，且无权重数据， 返回距离查询日期最近日期的权重信息；  
找不到对应日期的权重信息， 返回距离查询日期最近日期的权重信息；  

**示例**

In [8]:
#获取2018年5月9日这天的上证指数的成份股权重
df = get_index_weights(index_id="000001.XSHG", date="2018-05-09")
df.head()

,weight,display_name,date
code,,,
603648.XSHG,0.023,畅联股份,2018-05-09
603139.XSHG,0.007,康惠制药,2018-05-09
603138.XSHG,0.015,海量数据,2018-05-09
603136.XSHG,0.009,天目湖,2018-05-09
603131.XSHG,0.011,上海沪工,2018-05-09


## 行情数据
交易类数据提供指数的行情数据，通过API接口调用即可获取相应的数据。主要包括以下类别：

历史行情数据  
当前单位时间的行情数据

### 历史行情数据
获取指数历史交易数据，可以通过参数设置获取日k线、分钟k线数据。获取数据的基本属性如下：

open 时间段开始时价格  
close 时间段结束时价格  
low 最低价  
high 最高价  
volume 成交的指数数量  
money 成交的金额  
factor 前复权因子, 我们提供的价格都是前复权后的, 但是利用这个值可以算出原始价格, 方法是价格除以factor, 比如: close/factor  
high_limit 涨停价  
low_limit 跌停价  
avg 这段时间的平均价, 等于money/volume  
pre_close 前一个单位时间结束时的价格, 按天则是前一天的收盘价, 按分钟这是前一分钟的结束价格  
paused 布尔值, 这只指数是否停牌, 停牌时open/close/low/high/pre_close依然有值,都等于停牌前的收盘价, volume=money=0

**调用方法**

get_price(security, start_date='2015-01-01', end_date='2015-12-31', frequency='daily', fields=None, skip_paused=False, fq='pre')  
**注：设定不同的unit参数，获取日K线或分钟k线，详情见参数。** 这里请在使用时注意防止未来函数.

用户可以在 after_trading_end中调用get_price函数获取当日的开盘价、收盘价、成交额、成交量、最高价以及最低价等

**关于停牌: **因为此API可以获取多只股票的数据, 可能有的股票停牌有的没有, 为了保持时间轴的一致, 我们默认没有跳过停牌的日期, 停牌时使用停牌前的数据填充(请看[SecurityUnitData]的paused属性). 如想跳过, 请使用 skip_paused=True 参数, 同时只取一只股票的信息

**参数**

- security: 一支指数代码或者一个指数代码的list
- start_date: 字符串或者[datetime.datetime]/[datetime.date]对象, 开始时间, 默认是’2015-01-01’. 注意:

当取分钟数据时, 时间可以精确到分钟, 比如: 传入 datetime.datetime(2015, 1, 1, 10, 0, 0) 或者 '2015-01-01 10:00:00'.  
当取分钟数据时, 如果只传入日期, 则日内时间是当日的 00:00:00.  
当取天数据时, 传入的日内时间会被忽略  
- end_date: 格式同上, 结束时间, 默认是’2015-12-31’, 包含此日期. 注意: 当取分钟数据时, 如果 end_date 只有日期, 则日内时间等同于 00:00:00, 所以返回的数据是不包括 end_date 这一天的.
- frequency: 单位时间长度, 几天或者几分钟, 现在支持’Xd’,’Xm’, ‘daily’(等同于’1d’), ‘minute’(等同于’1m’), X是一个正整数, 分别表示X天和X分钟(不论是按天还是按分钟回测都能拿到这两种单位的数据), 注意, 当X > 1时, field只支持[‘open’, ‘close’, ‘high’, ‘low’, ‘volume’, ‘money’]这几个标准字段. 默认值是daily
- fields: 字符串list, 选择要获取的行情数据字段, 默认是None(表示[‘open’, ‘close’, ‘high’, ‘low’, ‘volume’, ‘money’]这几个标准字段), 支持属性里面的所有基本属性.
- skip_paused: 是否跳过不交易日期(包括停牌, 未上市或者退市后的日期). 如果不跳过, 停牌时会使用停牌前的数据填充(具体请看[SecurityUnitData]的paused属性), 上市前或者退市后数据都为 nan, , 但要注意:

默认为 False
- 当 skip_paused 是 True 时, 只能取一只股票的信息
- fq: 复权选项:

'pre': 前复权(根据’use_real_price’选项不同含义会有所不同, 参见set_option), 默认是前复权  
None: 不复权, 返回实际价格  
'post': 后复权  

**返回**

请注意, 为了方便比较一只指数的多个属性, 同时也满足对比多只指数的一个属性的需求, 我们在security参数是一只指数和多只指数时返回的结构完全不一样  
如果是一支指数, 则返回[pandas.DataFrame]对象, 行索引是[datetime.datetime]对象, 列索引是行情字段名字, 比如’open’/’close’. 比如: get_price('000300.XSHG')[:2]返回:

In [8]:
get_price('000300.XSHG')[:2]

,open,close,high,low,volume,money
2015-01-05,3566.09,3641.54,3669.04,3551.51,4.511981e+10,5.198498e+11
2015-01-06,3608.43,3641.06,3683.23,3587.23,4.209622e+10,4.985296e+11


如果是多支指数, 则返回[pandas.Panel]对象, 里面是很多[pandas.DataFrame]对象, 索引是行情字段(open/close/…), 每个[pandas.DataFrame]的行索引是[datetime.datetime]对象, 列索引是指数代号. 比如get_price(['000300.XSHG', '000001.XSHG'])['open'][:2]返回:

In [9]:
get_price(['000300.XSHG', '000001.XSHG'])['open'][:2]

,000300.XSHG,000001.XSHG
2015-01-05,3566.09,3258.63
2015-01-06,3608.43,3330.80


**示例**

In [11]:
# 获取一支指数
df = get_price('000001.XSHG') # 获取000001.XSHG的2015年的按天数据

In [13]:
# 获得000001.XSHG的2015年02月的分钟数据, 只获取open+close字段
df = get_price('000001.XSHG', start_date='2015-01-01', end_date='2015-02-01', frequency='minute', fields=['open', 'close']) 

In [14]:
# 获得000001.XSHG的2015年12月1号14:00-2015年12月2日12:00的分钟数据
df = get_price('000001.XSHG', start_date='2015-12-01 14:00:00', end_date='2015-12-02 12:00:00', frequency='1m') 

In [16]:
# 获取多只指数
panel =  get_price(get_index_stocks('000903.XSHG')) # 获取中证100的所有成分股的2015年的天数据, 返回一个[pandas.Panel]

In [26]:
df_open = panel['open']  # 获取开盘价的[pandas.DataFrame],  行索引是[datetime.datetime]对象, 列索引是指数代号

In [19]:
df_volume = panel['volume']  # 获取交易量的[pandas.DataFrame]

In [27]:
df_open['000001.XSHE'].head() # 获取上证指数的2015年每天的开盘价数据

2015-01-05    10.42
2015-01-06    10.33
2015-01-07    10.14
2015-01-08    10.10
2015-01-09     9.71
Name: 000001.XSHE, dtype: float64

# 沪深指数列表
我们支持近600种股票指数数据，包括指数的行情数据以及成分股数据。为了避免未来函数，我们支持获取历史任意时刻的指数成分股信息。  
温馨提示：

- 鉴于内容太多, 可使用Ctrl + F进行搜索)
- 000002.XSHG，上证A股指数的样本股是全部上海证券交易所上市A股，反映了上市A股的股价整体变动状况。


### 上海市场指数列表

| 指数代码    | 指数分类                     | 发布日期   | 行情开始日期 | 指数简写 |
| ----------- | ---------------------------- | ---------- | ------------ | -------- |
| 000001.XSHG | 上证指数                     | 1991-07-15 | 2005-01-04   | SZZS     |
| 000002.XSHG | A股指数                      | 1992-02-21 | 2005-01-04   | AGZS     |
| 000003.XSHG | B股指数                      | 1992-02-21 | 2005-01-04   | BGZS     |
| 000004.XSHG | 工业指数                     | 1993-05-03 | 2005-01-04   | GYZS     |
| 000005.XSHG | 商业指数                     | 1993-05-03 | 2005-01-04   | SYZS     |
| 000006.XSHG | 地产指数                     | 1993-05-03 | 2005-01-04   | DCZS     |
| 000007.XSHG | 公用指数                     | 1993-05-03 | 2005-01-04   | GYZS     |
| 000008.XSHG | 综合指数                     | 1993-05-03 | 2005-01-04   | ZHZS     |
| 000009.XSHG | 上证380                      | 2010-11-29 | 2010-11-29   | SZ380    |
| 000010.XSHG | 上证180                      | 2002-07-01 | 2005-01-04   | SZ180    |
| 000011.XSHG | 基金指数                     | 2000-06-09 | 2005-01-04   | JJZS     |
| 000012.XSHG | 国债指数                     | 2003-01-02 | 2005-01-04   | GZZS     |
| 000013.XSHG | 上证企业债指数               | 2003-06-09 | 2005-01-04   | QZZS     |
| 000015.XSHG | 红利指数                     | 2005-01-04 | 2005-01-04   | HLZS     |
| 000016.XSHG | 上证50                       | 2004-01-02 | 2005-01-04   | SZ50     |
| 000017.XSHG | 新综指                       | 2006-01-04 | 2006-01-04   | XZZ      |
| 000018.XSHG | 180金融                      | 2007-12-10 | 2007-12-10   | 180JR    |
| 000019.XSHG | 治理指数                     | 2008-01-02 | 2008-01-02   | ZLZS     |
| 000020.XSHG | 中型综指                     | 2008-05-12 | 2008-05-12   | ZXZZ     |
| 000021.XSHG | 180治理                      | 2008-09-10 | 2008-09-10   | 180ZL    |
| 000022.XSHG | 上证公司债指数               | 2008-11-19 | 2008-11-19   | HGSZ     |
| 000025.XSHG | 180基建                      | 2008-12-15 | 2008-12-15   | 180JJ    |
| 000026.XSHG | 180资源                      | 2008-12-15 | 2008-12-15   | 180ZY    |
| 000027.XSHG | 180运输                      | 2008-12-15 | 2008-12-15   | 180YS    |
| 000028.XSHG | 180成长                      | 2009-01-09 | 2009-01-09   | 180CZ    |
| 000029.XSHG | 180价值                      | 2009-01-09 | 2009-01-09   | 180JZ    |
| 000030.XSHG | 180R成长                     | 2009-01-09 | 2009-01-09   | 180RCZ   |
| 000031.XSHG | 180R价值                     | 2009-01-09 | 2009-01-09   | 180RJZ   |
| 000032.XSHG | 上证能源                     | 2009-01-09 | 2009-01-09   | SZNY     |
| 000033.XSHG | 上证材料                     | 2009-01-09 | 2009-01-09   | SZCL     |
| 000034.XSHG | 上证工业                     | 2009-01-09 | 2009-01-09   | SZGY     |
| 000035.XSHG | 上证可选                     | 2009-01-09 | 2009-01-09   | SZKX     |
| 000036.XSHG | 上证消费                     | 2009-01-09 | 2009-01-09   | SZXF     |
| 000037.XSHG | 上证医药                     | 2009-01-09 | 2009-01-09   | SZYY     |
| 000038.XSHG | 上证金融                     | 2009-01-09 | 2009-01-09   | SZJR     |
| 000039.XSHG | 上证信息                     | 2009-01-09 | 2009-01-09   | SZXX     |
| 000040.XSHG | 上证电信                     | 2009-01-09 | 2009-01-09   | SZDX     |
| 000041.XSHG | 上证公用                     | 2009-01-09 | 2009-01-09   | SZGY     |
| 000042.XSHG | 上证央企                     | 2009-03-30 | 2009-03-30   | SZYQ     |
| 000043.XSHG | 超大盘                       | 2009-04-23 | 2009-04-23   | CDP      |
| 000044.XSHG | 上证中盘                     | 2009-07-03 | 2009-07-03   | SZZP     |
| 000045.XSHG | 上证小盘                     | 2009-07-03 | 2009-07-03   | SZXP     |
| 000046.XSHG | 上证中小                     | 2009-07-03 | 2009-07-03   | SZZX     |
| 000047.XSHG | 上证全指                     | 2009-07-03 | 2009-07-03   | SZQZ     |
| 000048.XSHG | 责任指数                     | 2009-08-05 | 2009-08-05   | ZRZS     |
| 000049.XSHG | 上证民企                     | 2009-08-25 | 2009-08-25   | SZMQ     |
| 000050.XSHG | 50等权                       | 2011-01-04 | 2011-01-04   | 50DQ     |
| 000051.XSHG | 180等权                      | 2011-05-24 | 2011-05-24   | 180DQ    |
| 000052.XSHG | 50基本                       | 2012-01-09 | 2012-01-09   | 50JB     |
| 000053.XSHG | 180基本                      | 2012-01-09 | 2012-01-09   | 180JB    |
| 000054.XSHG | 上证海外                     | 2010-01-04 | 2010-01-04   | SZHW     |
| 000055.XSHG | 上证地企                     | 2010-01-04 | 2010-01-04   | SZDQ     |
| 000056.XSHG | 上证国企                     | 2010-01-04 | 2010-01-04   | SZGQ     |
| 000057.XSHG | 全指成长                     | 2010-01-04 | 2010-01-04   | QZCZ     |
| 000058.XSHG | 全指价值                     | 2010-01-04 | 2010-01-04   | QZJZ     |
| 000059.XSHG | 全R成长                      | 2010-01-04 | 2010-01-04   | QRCZ     |
| 000060.XSHG | 全R价值                      | 2010-01-04 | 2010-01-04   | QRJZ     |
| 000061.XSHG | 沪企债30                     | 2010-01-04 | 2010-01-04   | HQZ30    |
| 000062.XSHG | 上证沪企                     | 2010-01-04 | 2010-01-04   | SZHQ     |
| 000063.XSHG | 上证周期                     | 2010-02-03 | 2010-02-03   | SZZQ     |
| 000064.XSHG | 非周期                       | 2010-02-03 | 2010-02-03   | FZQ      |
| 000065.XSHG | 上证龙头                     | 2010-02-09 | 2010-02-09   | SZLT     |
| 000066.XSHG | 上证商品                     | 2010-04-30 | 2010-04-30   | SZSP     |
| 000067.XSHG | 上证新兴                     | 2010-04-30 | 2010-04-30   | SZXX     |
| 000068.XSHG | 上证资源                     | 2010-05-28 | 2010-05-28   | SZZY     |
| 000069.XSHG | 消费80                       | 2010-05-28 | 2010-05-28   | XF80     |
| 000070.XSHG | 能源等权                     | 2010-08-18 | 2010-08-18   | NYDQ     |
| 000071.XSHG | 材料等权                     | 2010-08-18 | 2010-08-18   | CLDQ     |
| 000072.XSHG | 工业等权                     | 2010-08-18 | 2010-08-18   | GYDQ     |
| 000073.XSHG | 可选等权                     | 2010-08-18 | 2010-08-18   | KXDQ     |
| 000074.XSHG | 消费等权                     | 2010-08-18 | 2010-08-18   | XFDQ     |
| 000075.XSHG | 医药等权                     | 2010-08-18 | 2010-08-18   | YYDQ     |
| 000076.XSHG | 金融等权                     | 2010-08-18 | 2010-08-18   | JRDQ     |
| 000077.XSHG | 信息等权                     | 2010-08-18 | 2010-08-18   | XXDQ     |
| 000078.XSHG | 电信等权                     | 2010-08-18 | 2010-08-18   | DXDQ     |
| 000079.XSHG | 公用等权                     | 2010-08-18 | 2010-08-18   | GYDQ     |
| 000090.XSHG | 上证流通                     | 2010-12-02 | 2010-12-02   | SZLT     |
| 000091.XSHG | 沪财中小                     | 2011-04-06 | 2011-04-06   | HCZX     |
| 000092.XSHG | 资源50                       | 2011-05-10 | 2011-05-10   | ZY50     |
| 000093.XSHG | 180分层                      | 2011-05-24 | 2011-05-24   | 180FC    |
| 000094.XSHG | 上证上游                     | 2011-05-10 | 2011-05-10   | SZSY     |
| 000095.XSHG | 上证中游                     | 2011-05-10 | 2011-05-10   | SZZY     |
| 000096.XSHG | 上证下游                     | 2011-05-10 | 2011-05-10   | SZXY     |
| 000097.XSHG | 高端装备                     | 2011-05-24 | 2011-05-24   | GDZB     |
| 000098.XSHG | 上证F200                     | 2011-07-01 | 2011-07-01   | SZF200   |
| 000099.XSHG | 上证F300                     | 2011-07-01 | 2011-07-01   | SZF300   |
| 000100.XSHG | 上证F500                     | 2011-07-01 | 2011-07-01   | SZF500   |
| 000101.XSHG | 5年信用                      | 2013-04-01 | 2013-04-01   | 5NXY     |
| 000102.XSHG | 沪投资品                     | 2011-06-13 | 2011-06-13   | HTZP     |
| 000103.XSHG | 沪消费品                     | 2011-06-13 | 2011-06-13   | HXFP     |
| 000104.XSHG | 380能源                      | 2011-06-13 | 2011-06-13   | 380NY    |
| 000105.XSHG | 380材料                      | 2011-06-13 | 2011-06-13   | 380CL    |
| 000106.XSHG | 380工业                      | 2011-06-13 | 2011-06-13   | 380GY    |
| 000107.XSHG | 380可选                      | 2011-06-13 | 2011-06-13   | 380KX    |
| 000108.XSHG | 380消费                      | 2011-06-13 | 2011-06-13   | 380XF    |
| 000109.XSHG | 380医药                      | 2011-06-13 | 2011-06-13   | 380YY    |
| 000110.XSHG | 380金融                      | 2011-06-13 | 2011-06-13   | 380JR    |
| 000111.XSHG | 380信息                      | 2011-06-13 | 2011-06-13   | 380XX    |
| 000112.XSHG | 380电信                      | 2011-06-13 | 2011-06-13   | 380DX    |
| 000113.XSHG | 380公用                      | 2011-06-13 | 2011-06-13   | 380GY    |
| 000114.XSHG | 持续产业                     | 2011-08-22 | 2011-08-22   | CXCY     |
| 000115.XSHG | 380等权                      | 2011-08-12 | 2011-08-12   | 380DQ    |
| 000116.XSHG | 信用100                      | 2013-02-26 | 2013-02-26   | XY100    |
| 000117.XSHG | 380成长                      | 2011-10-18 | 2011-10-18   | 380CZ    |
| 000118.XSHG | 380价值                      | 2011-10-18 | 2011-10-18   | 380JZ    |
| 000119.XSHG | 380R成长                     | 2011-10-18 | 2011-10-18   | 380RCZ   |
| 000120.XSHG | 380R价值                     | 2011-10-18 | 2011-10-18   | 380RJZ   |
| 000121.XSHG | 医药主题                     | 2011-10-18 | 2011-10-18   | YYZT     |
| 000122.XSHG | 农业主题                     | 2011-10-18 | 2011-10-18   | NYZT     |
| 000123.XSHG | 180动态                      | 2013-02-08 | 2013-02-08   | 180DT    |
| 000125.XSHG | 180稳定                      | 2013-02-08 | 2013-02-08   | 180WD    |
| 000126.XSHG | 消费50                       | 2011-12-09 | 2011-12-09   | XF50     |
| 000128.XSHG | 380基本                      | 2012-01-09 | 2012-01-09   | 380JB    |
| 000129.XSHG | 180波动                      | 2012-01-09 | 2012-01-09   | 180BD    |
| 000130.XSHG | 380波动                      | 2012-01-09 | 2012-01-09   | 380BD    |
| 000131.XSHG | 上证高新                     | 2012-02-10 | 2012-02-10   | SZGX     |
| 000132.XSHG | 上证100                      | 2012-04-20 | 2012-04-20   | SZ100    |
| 000133.XSHG | 上证150                      | 2012-04-20 | 2012-04-20   | SZ150    |
| 000134.XSHG | 上证银行                     | 2012-05-29 | 2012-05-29   | SZYH     |
| 000135.XSHG | 180高贝                      | 2012-08-06 | 2012-08-06   | 180GB    |
| 000136.XSHG | 180低贝                      | 2012-08-06 | 2012-08-06   | 180DB    |
| 000137.XSHG | 380高贝                      | 2012-08-06 | 2012-08-06   | 380GB    |
| 000138.XSHG | 380低贝                      | 2012-08-06 | 2012-08-06   | 380DB    |
| 000139.XSHG | 上证转债                     | 2012-09-12 | 2012-09-12   | SZZZ     |
| 000141.XSHG | 380动态                      | 2013-02-08 | 2013-02-08   | 380DT    |
| 000142.XSHG | 380稳定                      | 2013-02-08 | 2013-02-08   | 380WD    |
| 000145.XSHG | 优势资源                     | 2012-05-29 | 2012-05-29   | YSZY     |
| 000146.XSHG | 优势制造                     | 2012-05-29 | 2012-05-29   | YSZZ     |
| 000147.XSHG | 优势消费                     | 2012-05-29 | 2012-05-29   | YSXF     |
| 000148.XSHG | 消费领先                     | 2012-05-29 | 2012-05-29   | XFLX     |
| 000149.XSHG | 180红利                      | 2012-07-20 | 2012-07-20   | 180HL    |
| 000150.XSHG | 380红利                      | 2012-07-20 | 2012-07-20   | 380HL    |
| 000151.XSHG | 上国红利                     | 2012-07-20 | 2012-07-20   | SGHL     |
| 000152.XSHG | 上央红利                     | 2012-07-20 | 2012-07-20   | SYHL     |
| 000153.XSHG | 上民红利                     | 2012-07-20 | 2012-07-20   | SMHL     |
| 000155.XSHG | 市值百强                     | 2012-07-20 | 2012-07-20   | SZBQ     |
| 000158.XSHG | 上证环保                     | 2012-09-25 | 2012-09-25   | SZHB     |
| 000159.XSHG | 上证沪股通指数               | 2014-11-17 | 2014-11-17   | HGT      |
| 000160.XSHG | 上证一带一路主题指数         | 2015-06-24 | 2015-06-24   | HXSL     |
| 000161.XSHG | 上证中国制造2025主题指数     | 2015-10-29 | 2015-10-29   | HZGZ     |
| 000162.XSHG | 上证互联网+主题指数          | 2015-10-29 | 2015-10-29   | HHL+     |
| 000171.XSHG | 新兴成指                     | 2017-01-25 | 2017-01-25   | XXCZ     |
| 000188.XSHG | 中国波指                     | 2016-11-28 | 2016-11-28   | ZZBZ     |
| 000300.XSHG | 沪深300                      | 2005-04-08 | 2005-04-08   | HS300    |
| 000801.XSHG | 资源80                       | 2011-11-08 | 2011-11-08   | ZY80     |
| 000802.XSHG | 500沪市                      | 2011-11-08 | 2011-11-08   | 500HS    |
| 000803.XSHG | 300波动                      | 2012-01-09 | 2012-01-09   | 300BD    |
| 000804.XSHG | 500波动                      | 2012-01-09 | 2012-01-09   | 500BD    |
| 000805.XSHG | A股资源                      | 2012-01-06 | 2012-01-06   | AGZY     |
| 000806.XSHG | 消费服务                     | 2012-02-10 | 2012-02-10   | XFFW     |
| 000807.XSHG | 食品饮料                     | 2012-02-17 | 2012-02-17   | SPYL     |
| 000808.XSHG | 医药生物                     | 2012-02-17 | 2012-02-17   | YYSW     |
| 000809.XSHG | 细分农业                     | 2012-04-11 | 2012-04-11   | XFNY     |
| 000810.XSHG | 细分能源                     | 2012-04-11 | 2012-04-11   | XFNY     |
| 000811.XSHG | 细分有色                     | 2012-04-11 | 2012-04-11   | XFYS     |
| 000812.XSHG | 细分机械                     | 2012-04-11 | 2012-04-11   | XFJX     |
| 000813.XSHG | 细分化工                     | 2012-04-11 | 2012-04-11   | XFHG     |
| 000814.XSHG | 细分医药                     | 2012-04-11 | 2012-04-11   | XFYY     |
| 000815.XSHG | 细分食品                     | 2012-04-11 | 2012-04-11   | XFSP     |
| 000816.XSHG | 细分地产                     | 2012-04-11 | 2012-04-11   | XFDC     |
| 000817.XSHG | 兴证海峡                     | 2011-09-27 | 2011-09-27   | XZHX     |
| 000818.XSHG | 细分金融                     | 2012-04-11 | 2012-04-11   | XFJR     |
| 000819.XSHG | 有色金属                     | 2012-05-09 | 2012-05-09   | YSJS     |
| 000820.XSHG | 煤炭指数                     | 2012-05-09 | 2012-05-09   | MTZS     |
| 000821.XSHG | 300红利                      | 2012-07-20 | 2012-07-20   | 300HL    |
| 000822.XSHG | 500红利                      | 2012-07-20 | 2012-07-20   | 500HL    |
| 000823.XSHG | 中证800有色金属指数          | 2012-12-21 | 2012-12-21   | 800YS    |
| 000824.XSHG | 国企红利                     | 2012-07-20 | 2012-07-20   | GQHL     |
| 000825.XSHG | 央企红利                     | 2012-07-20 | 2012-07-20   | YQHL     |
| 000826.XSHG | 民企红利                     | 2012-07-20 | 2012-07-20   | MQHL     |
| 000827.XSHG | 中证环保                     | 2012-09-25 | 2012-09-25   | ZZHB     |
| 000828.XSHG | 300高贝                      | 2012-08-06 | 2012-08-06   | 300GB    |
| 000829.XSHG | 300低贝                      | 2012-08-06 | 2012-08-06   | 300DB    |
| 000830.XSHG | 500高贝                      | 2012-08-06 | 2012-08-06   | 500GB    |
| 000831.XSHG | 500低贝                      | 2012-08-06 | 2012-08-06   | 500DB    |
| 000832.XSHG | 中证转债                     | 2012-09-12 | 2012-09-12   | ZZZZ     |
| 000833.XSHG | 中高企债                     | 2012-09-10 | 2012-09-10   | ZGQZ     |
| 000838.XSHG | 创业价值                     | 2012-09-26 | 2012-09-26   | CYJZ     |
| 000839.XSHG | 浙企综指                     | 2012-11-09 | 2012-11-09   | ZQZZ     |
| 000840.XSHG | 浙江民企                     | 2012-11-09 | 2012-11-09   | ZJMQ     |
| 000841.XSHG | 800医药                      | 2012-12-21 | 2012-12-21   | 800YY    |
| 000842.XSHG | 800等权                      | 2012-12-21 | 2012-12-21   | 800DQ    |
| 000843.XSHG | 300动态                      | 2013-01-07 | 2013-01-07   | 300DT    |
| 000844.XSHG | 300稳定                      | 2013-01-07 | 2013-01-07   | 300WD    |
| 000846.XSHG | ESG100                       | 2012-10-16 | 2012-10-16   | ESG100   |
| 000847.XSHG | 中证腾安价值100指数          | 2013-11-18 | 2013-11-18   | ZZTA     |
| 000849.XSHG | 沪深300非银行金融指数        | 2012-12-21 | 2012-12-21   | 300FY    |
| 000850.XSHG | 沪深300有色金属指数          | 2012-12-21 | 2012-12-21   | 300YS    |
| 000851.XSHG | 中证百度百发策略100指数      | 2014-07-30 | 2014-07-30   | BF100    |
| 000852.XSHG | 中证1000指数                 | 2014-10-17 | 2014-10-17   | ZZ1000   |
| 000853.XSHG | 中证申万一带一路主题投资指数 | 2015-04-23 | 2015-04-23   | CSSWSL   |
| 000854.XSHG | 500原料                      | 2013-11-06 | 2013-11-06   | 500YL    |
| 000855.XSHG | 央视财经500指数              | 2014-11-03 | 2014-11-03   | YS500    |
| 000856.XSHG | 500工业                      | 2013-11-06 | 2013-11-06   | 500GY    |
| 000857.XSHG | 500医药                      | 2013-11-06 | 2013-11-06   | 500YY    |
| 000858.XSHG | 500信息                      | 2013-11-06 | 2013-11-06   | 500XX    |
| 000891.XSHG | 中国战略新兴产业综合指数     | 2017-01-25 | 2017-01-25   | XXZZ     |
| 000901.XSHG | 小康指数                     | 2010-04-01 | 2010-04-01   | XKZS     |
| 000902.XSHG | 中证流通                     | 2006-02-27 | 2006-02-27   | ZZLT     |
| 000903.XSHG | 中证100                      | 2006-05-29 | 2006-05-29   | ZZ100    |
| 000904.XSHG | 中证200                      | 2007-01-15 | 2007-01-15   | ZZ200    |
| 000905.XSHG | 中证500                      | 2007-01-15 | 2007-01-15   | ZZ500    |
| 000906.XSHG | 中证800                      | 2007-01-15 | 2007-01-15   | ZZ800    |
| 000907.XSHG | 中证700                      | 2007-01-15 | 2007-01-15   | ZZ700    |
| 000908.XSHG | 300能源                      | 2007-07-02 | 2007-07-02   | 300NY    |
| 000909.XSHG | 300材料                      | 2007-07-02 | 2007-07-02   | 300CL    |
| 000910.XSHG | 300工业                      | 2007-07-02 | 2007-07-02   | 300GY    |
| 000911.XSHG | 300可选                      | 2007-07-02 | 2007-07-02   | 300KX    |
| 000912.XSHG | 300消费                      | 2007-07-02 | 2007-07-02   | 300XF    |
| 000913.XSHG | 300医药                      | 2007-07-02 | 2007-07-02   | 300YY    |
| 000914.XSHG | 300金融                      | 2007-07-02 | 2007-07-02   | 300JR    |
| 000915.XSHG | 300信息                      | 2007-07-02 | 2007-07-02   | 300XX    |
| 000916.XSHG | 300电信                      | 2007-07-02 | 2007-07-02   | 300DX    |
| 000917.XSHG | 300公用                      | 2007-07-02 | 2007-07-02   | 300GY    |
| 000918.XSHG | 300成长                      | 2008-01-21 | 2008-01-21   | 300CZ    |
| 000919.XSHG | 300价值                      | 2008-01-21 | 2008-01-21   | 300JZ    |
| 000920.XSHG | 300R成长                     | 2008-01-21 | 2008-01-21   | 300RCZ   |
| 000921.XSHG | 300R价值                     | 2008-01-21 | 2008-01-21   | 300RJZ   |
| 000922.XSHG | 中证红利                     | 2008-05-26 | 2008-05-26   | ZZHL     |
| 000923.XSHG | 公司债                       | 2008-11-19 | 2008-11-19   | GSZ      |
| 000925.XSHG | 基本面50                     | 2009-02-26 | 2009-02-26   | JBM50    |
| 000926.XSHG | 中证央企                     | 2009-03-30 | 2009-03-30   | ZZYQ     |
| 000927.XSHG | 央企100                      | 2009-03-30 | 2009-03-30   | YQ100    |
| 000928.XSHG | 中证能源                     | 2009-07-03 | 2009-07-03   | ZZNY     |
| 000929.XSHG | 中证材料                     | 2009-07-03 | 2009-07-03   | ZZCL     |
| 000930.XSHG | 中证工业                     | 2009-07-03 | 2009-07-03   | ZZGY     |
| 000931.XSHG | 中证可选                     | 2009-07-03 | 2009-07-03   | ZZKX     |
| 000932.XSHG | 中证消费                     | 2009-07-03 | 2009-07-03   | ZZXF     |
| 000933.XSHG | 中证医药                     | 2009-07-03 | 2009-07-03   | ZZYY     |
| 000934.XSHG | 中证金融                     | 2009-07-03 | 2009-07-03   | ZZJR     |
| 000935.XSHG | 中证信息                     | 2009-07-03 | 2009-07-03   | ZZXX     |
| 000936.XSHG | 中证电信                     | 2009-07-03 | 2009-07-03   | ZZDX     |
| 000937.XSHG | 中证公用                     | 2009-07-03 | 2009-07-03   | ZZGY     |
| 000938.XSHG | 中证民企                     | 2009-08-25 | 2009-08-25   | ZZMQ     |
| 000939.XSHG | 民企200                      | 2009-08-25 | 2009-08-25   | MQ200    |
| 000940.XSHG | 财富大盘                     | 2009-09-25 | 2009-09-25   | CFDP     |
| 000941.XSHG | 新能源                       | 2009-10-28 | 2009-10-28   | XNY      |
| 000942.XSHG | 内地消费                     | 2009-10-28 | 2009-10-28   | NDXF     |
| 000943.XSHG | 内地基建                     | 2009-10-28 | 2009-10-28   | NDJJ     |
| 000944.XSHG | 内地资源                     | 2009-10-28 | 2009-10-28   | NDZY     |
| 000945.XSHG | 内地运输                     | 2009-10-28 | 2009-10-28   | NDYS     |
| 000946.XSHG | 内地金融                     | 2009-10-28 | 2009-10-28   | NDJR     |
| 000947.XSHG | 内地银行                     | 2009-10-28 | 2009-10-28   | NDYH     |
| 000948.XSHG | 内地地产                     | 2009-10-28 | 2009-10-28   | NDDC     |
| 000949.XSHG | 内地农业                     | 2009-10-28 | 2009-10-28   | NDNY     |
| 000950.XSHG | 300基建                      | 2009-10-28 | 2009-10-28   | 300JJ    |
| 000951.XSHG | 300银行                      | 2009-10-28 | 2009-10-28   | 300YH    |
| 000952.XSHG | 300地产                      | 2009-10-28 | 2009-10-28   | 300DC    |
| 000953.XSHG | 中证地企                     | 2010-02-09 | 2010-02-09   | ZZDQ     |
| 000954.XSHG | 地企100                      | 2010-02-09 | 2010-02-09   | DQ100    |
| 000955.XSHG | 中证国企                     | 2010-02-09 | 2010-02-09   | ZZGQ     |
| 000956.XSHG | 国企200                      | 2010-02-09 | 2010-02-09   | GQ200    |
| 000957.XSHG | 300运输                      | 2009-06-16 | 2009-06-16   | 300YS    |
| 000958.XSHG | 创业成长                     | 2010-03-24 | 2010-03-24   | CYCZ     |
| 000959.XSHG | 银河99                       | 2010-02-09 | 2010-02-09   | YH99     |
| 000960.XSHG | 中证龙头                     | 2010-02-09 | 2010-02-09   | ZZLT     |
| 000961.XSHG | 中证上游                     | 2010-04-16 | 2010-04-16   | ZZSY     |
| 000962.XSHG | 中证中游                     | 2010-04-16 | 2010-04-16   | ZZZY     |
| 000963.XSHG | 中证下游                     | 2010-04-16 | 2010-04-16   | ZZXY     |
| 000964.XSHG | 中证新兴                     | 2010-04-30 | 2010-04-30   | ZZXX     |
| 000965.XSHG | 基本200                      | 2010-06-02 | 2010-06-02   | JB200    |
| 000966.XSHG | 基本400                      | 2010-06-02 | 2010-06-02   | JB400    |
| 000967.XSHG | 基本600                      | 2010-06-02 | 2010-06-02   | JB600    |
| 000968.XSHG | 300周期                      | 2010-05-28 | 2010-05-28   | 300ZQ    |
| 000969.XSHG | 300非周                      | 2010-05-28 | 2010-05-28   | 300FZ    |
| 000970.XSHG | ESG40                        | 2010-09-17 | 2010-09-17   | ESG40    |
| 000971.XSHG | 等权90                       | 2010-12-02 | 2010-12-02   | DQ90     |
| 000972.XSHG | 300沪市                      | 2010-12-02 | 2010-12-02   | 300HS    |
| 000973.XSHG | 技术领先                     | 2011-01-04 | 2011-01-04   | JSLX     |
| 000974.XSHG | 中证800金融指数              | 2013-05-09 | 2013-05-09   | 800JR    |
| 000975.XSHG | 钱江30                       | 2014-01-02 | 2014-01-02   | QJ30     |
| 000976.XSHG | 新华金牛                     | 2014-02-18 | 2014-02-18   | XHJN     |
| 000977.XSHG | 内地低碳                     | 2011-01-21 | 2011-01-21   | NDDT     |
| 000978.XSHG | 医药100                      | 2011-03-18 | 2011-03-18   | YY100    |
| 000979.XSHG | 大宗商品                     | 2011-08-22 | 2011-08-22   | DZSP     |
| 000980.XSHG | 中证超大                     | 2011-05-10 | 2011-05-10   | ZZCD     |
| 000981.XSHG | 300分层                      | 2011-06-13 | 2011-06-13   | 300FC    |
| 000982.XSHG | 500等权                      | 2011-06-13 | 2011-06-13   | 500DQ    |
| 000983.XSHG | 智能资产                     | 2011-06-17 | 2011-06-17   | ZNZC     |
| 000984.XSHG | 300等权                      | 2011-08-02 | 2011-08-02   | 300DQ    |
| 000985.XSHG | 中证全指                     | 2011-08-02 | 2011-08-02   | ZZQZ     |
| 000986.XSHG | 全指能源                     | 2011-08-02 | 2011-08-02   | QZNY     |
| 000987.XSHG | 全指材料                     | 2011-08-02 | 2011-08-02   | QZCL     |
| 000988.XSHG | 全指工业                     | 2011-08-02 | 2011-08-02   | QZGY     |
| 000989.XSHG | 全指可选                     | 2011-08-02 | 2011-08-02   | QZKX     |
| 000990.XSHG | 全指消费                     | 2011-08-02 | 2011-08-02   | QZXF     |
| 000991.XSHG | 全指医药                     | 2011-08-02 | 2011-08-02   | QZYY     |
| 000992.XSHG | 全指金融                     | 2011-08-02 | 2011-08-02   | QZJR     |
| 000993.XSHG | 全指信息                     | 2011-08-02 | 2011-08-02   | QZXX     |
| 000994.XSHG | 全指电信                     | 2011-08-02 | 2011-08-02   | QZDX     |
| 000995.XSHG | 全指公用                     | 2011-08-02 | 2011-08-02   | QZGY     |
| 000996.XSHG | 领先行业                     | 2011-08-02 | 2011-08-02   | LXHY     |
| 000997.XSHG | 大消费                       | 2011-11-11 | 2011-11-11   | DXF      |
| 000998.XSHG | 中证TMT                      | 2011-11-08 | 2011-11-08   | ZZTMT    |
| 000999.XSHG | 中证两岸三地500指数          | 2010-01-18 | 2010-01-18   | LASD     |

 

## 深圳市场指数列表

| 指数代码    | 指数分类                       | 发布日期   | 行情开始日期 | 指数简写 |
| ----------- | ------------------------------ | ---------- | ------------ | -------- |
| 399001.XSHE | 深证成指                       | 1995-01-23 | 2005-01-04   | SZCZ     |
| 399002.XSHE | 深成指R                        | 1995-01-23 | 2005-01-04   | SCZR     |
| 399003.XSHE | 成份B指                        | 1995-01-23 | 2005-01-04   | CFBZ     |
| 399004.XSHE | 深证100R                       | 2003-01-02 | 2005-01-04   | SZ100R   |
| 399005.XSHE | 中小板指                       | 2006-01-24 | 2006-01-24   | ZXBZ     |
| 399006.XSHE | 创业板指                       | 2010-06-01 | 2010-06-01   | CYBZ     |
| 399007.XSHE | 深证300                        | 2009-11-04 | 2009-11-04   | SZ300    |
| 399008.XSHE | 中小300                        | 2010-03-22 | 2010-03-22   | ZX300    |
| 399009.XSHE | 深证200                        | 2011-09-01 | 2011-09-01   | SZ200    |
| 399010.XSHE | 深证700                        | 2011-09-01 | 2011-09-01   | SZ700    |
| 399011.XSHE | 深证1000                       | 2011-09-01 | 2011-09-01   | SZ1000   |
| 399012.XSHE | 创业300                        | 2013-01-07 | 2013-01-07   | CY300    |
| 399013.XSHE | 深市精选                       | 2014-05-30 | 2014-05-30   | SSJX     |
| 399015.XSHE | 深证中小创新指数               | 2015-03-24 | 2015-03-24   | ZXCX     |
| 399016.XSHE | 深证创新指数                   | 2016-12-01 | 2016-12-01   | SZCX     |
| 399017.XSHE | 中小板创新指数                 | 2016-12-01 | 2016-12-01   | SMECX    |
| 399018.XSHE | 创业板创新指数                 | 2016-12-01 | 2016-12-01   | CYCX     |
| 399100.XSHE | 新指数                         | 2006-02-16 | 2006-02-16   | XZS      |
| 399101.XSHE | 中小板综                       | 2005-12-01 | 2005-12-01   | ZXBZ     |
| 399102.XSHE | 创业板综合指数                 | 2010-08-20 | 2010-08-20   | CYBZ     |
| 399103.XSHE | 乐富指数                       | 2011-12-02 | 2011-12-02   | LFZS     |
| 399106.XSHE | 深证综指                       | 1991-04-04 | 2005-01-04   | SZZZ     |
| 399107.XSHE | 深证A指                        | 1992-10-04 | 2005-01-04   | SZAZ     |
| 399108.XSHE | 深证B指                        | 1992-10-06 | 2005-01-04   | SZBZ     |
| 399231.XSHE | 农林指数                       | 2013-03-04 | 2013-03-04   | NLZS     |
| 399232.XSHE | 采矿指数                       | 2013-03-04 | 2013-03-04   | CKZS     |
| 399233.XSHE | 制造指数                       | 2013-03-04 | 2013-03-04   | ZZZS     |
| 399234.XSHE | 水电指数                       | 2013-03-04 | 2013-03-04   | SDZS     |
| 399235.XSHE | 建筑指数                       | 2013-03-04 | 2013-03-04   | JZZS     |
| 399236.XSHE | 批零指数                       | 2013-03-04 | 2013-03-04   | PLZS     |
| 399237.XSHE | 运输指数                       | 2013-03-04 | 2013-03-04   | YSZS     |
| 399238.XSHE | 餐饮指数                       | 2013-03-04 | 2013-03-04   | CYZS     |
| 399239.XSHE | IT指数                         | 2013-03-04 | 2013-03-04   | ITZS     |
| 399240.XSHE | 金融指数                       | 2013-03-04 | 2013-03-04   | JRZS     |
| 399241.XSHE | 地产指数                       | 2013-03-04 | 2013-03-04   | DCZS     |
| 399242.XSHE | 商务指数                       | 2013-03-04 | 2013-03-04   | SWZS     |
| 399243.XSHE | 科研指数                       | 2013-03-04 | 2013-03-04   | KYZS     |
| 399244.XSHE | 公共指数                       | 2013-03-04 | 2013-03-04   | GGZS     |
| 399248.XSHE | 文化指数                       | 2013-03-04 | 2013-03-04   | WHZS     |
| 399249.XSHE | 综企指数                       | 2013-03-04 | 2013-03-04   | ZQZS     |
| 399298.XSHE | 深证中高等级信用债指数         | 2014-09-25 | 2014-09-25   | SXZG     |
| 399299.XSHE | 深证中低等级信用债指数         | 2014-09-25 | 2014-09-25   | SXZD     |
| 399300.XSHE | 沪深300                        | 2005-04-08 | 2005-04-08   | HS300    |
| 399301.XSHE | 深信用债                       | 2013-01-11 | 2013-01-11   | SXYZ     |
| 399302.XSHE | 深公司债                       | 2013-01-11 | 2013-01-11   | SGSZ     |
| 399303.XSHE | 国证2000                       | 2014-03-28 | 2014-03-28   | GZ2000   |
| 399305.XSHE | 基金指数                       | 2000-07-03 | 2005-01-04   | JJZS     |
| 399306.XSHE | 深证ETF                        | 2011-12-02 | 2011-12-02   | SZETF    |
| 399307.XSHE | 深证转债                       | 2014-08-27 | 2014-08-27   | SZZZ     |
| 399310.XSHE | 国证50                         | 2005-09-05 | 2005-09-05   | GZ50     |
| 399311.XSHE | 国证1000                       | 2005-02-03 | 2005-02-03   | GZ1000   |
| 399312.XSHE | 国证300                        | 2005-02-03 | 2005-02-03   | GZ300    |
| 399313.XSHE | 巨潮100                        | 2005-02-03 | 2005-02-03   | JC100    |
| 399314.XSHE | 巨潮大盘                       | 2005-02-03 | 2005-02-03   | JCDP     |
| 399315.XSHE | 巨潮中盘                       | 2005-02-03 | 2005-02-03   | JCZP     |
| 399316.XSHE | 巨潮小盘                       | 2005-02-03 | 2005-02-03   | JCXP     |
| 399317.XSHE | 国证A指                        | 2005-02-03 | 2005-02-03   | GZAZ     |
| 399318.XSHE | 巨潮B股指数                    | 2005-02-03 | 2005-02-03   | JCBZ     |
| 399319.XSHE | 资源优势                       | 2005-01-04 | 2005-01-04   | ZYYS     |
| 399320.XSHE | 国证服务                       | 2005-01-04 | 2005-01-04   | GZFW     |
| 399321.XSHE | 国证红利                       | 2005-01-04 | 2005-01-04   | GZHL     |
| 399322.XSHE | 国证治理                       | 2005-12-12 | 2005-12-12   | GZZL     |
| 399324.XSHE | 深证红利                       | 2006-01-24 | 2006-01-24   | SZHL     |
| 399326.XSHE | 成长40                         | 2006-01-24 | 2006-01-24   | CZ40     |
| 399328.XSHE | 深证治理                       | 2006-01-24 | 2006-01-24   | SZZL     |
| 399330.XSHE | 深证100                        | 2006-01-24 | 2006-01-24   | SZ100    |
| 399332.XSHE | 深证创新(旧)                   | 2006-02-27 | 2006-02-27   | SZCX     |
| 399333.XSHE | 中小板R                        | 2006-12-27 | 2006-12-27   | ZXBR     |
| 399335.XSHE | 深证央企                       | 2009-08-03 | 2009-08-03   | SZYQ     |
| 399337.XSHE | 深证民营                       | 2009-08-03 | 2009-08-03   | SZMY     |
| 399339.XSHE | 深证科技                       | 2009-08-03 | 2009-08-03   | SZKJ     |
| 399341.XSHE | 深证责任                       | 2009-08-03 | 2009-08-03   | SZZR     |
| 399344.XSHE | 深证300R                       | 2009-11-04 | 2009-11-04   | SZ300R   |
| 399346.XSHE | 深证成长                       | 2009-11-04 | 2009-11-04   | SZCZ     |
| 399348.XSHE | 深证价值                       | 2009-11-04 | 2009-11-04   | SZJZ     |
| 399350.XSHE | 皖江30                         | 2012-12-18 | 2012-12-18   | WJ30     |
| 399351.XSHE | 深报指数                       | 2004-09-01 | 2005-01-04   | SBZS     |
| 399352.XSHE | 深报综指                       | 2005-02-03 | 2005-02-03   | SBZZ     |
| 399353.XSHE | 国证物流                       | 2005-03-01 | 2005-03-01   | GZWL     |
| 399354.XSHE | 分析师指数                     | 2006-11-25 | 2006-11-25   | FXSZS    |
| 399355.XSHE | 长三角                         | 2007-07-02 | 2007-07-02   | ZSJ      |
| 399356.XSHE | 珠三角                         | 2007-07-02 | 2007-07-02   | ZSJ      |
| 399357.XSHE | 环渤海                         | 2007-07-02 | 2007-07-02   | HBH      |
| 399358.XSHE | 泰达指数                       | 2008-01-02 | 2008-01-02   | TDZS     |
| 399359.XSHE | 国证基建                       | 2009-08-03 | 2009-08-03   | GZJJ     |
| 399360.XSHE | 国证装备                       | 2009-08-03 | 2009-08-03   | GZZB     |
| 399361.XSHE | 国证商业                       | 2009-08-03 | 2009-08-03   | GZSY     |
| 399362.XSHE | 国证民营                       | 2009-08-03 | 2009-08-03   | GZMY     |
| 399363.XSHE | 计算机指                       | 2009-08-03 | 2009-08-03   | JSJZ     |
| 399364.XSHE | 中金消费                       | 2009-08-03 | 2009-08-03   | ZJXF     |
| 399365.XSHE | 国证农业                       | 2009-11-04 | 2009-11-04   | GZNY     |
| 399366.XSHE | 国证大宗                       | 2009-11-04 | 2009-11-04   | GZDZ     |
| 399367.XSHE | 巨潮地产                       | 2009-11-04 | 2009-11-04   | JCDC     |
| 399368.XSHE | 国证军工                       | 2009-11-04 | 2009-11-04   | GZJG     |
| 399369.XSHE | CBN-兴全                       | 2009-11-04 | 2009-11-04   | CBNXQ    |
| 399370.XSHE | 国证成长                       | 2010-01-04 | 2010-01-04   | GZCZ     |
| 399371.XSHE | 国证价值                       | 2010-01-04 | 2010-01-04   | GZJZ     |
| 399372.XSHE | 大盘成长                       | 2010-01-04 | 2010-01-04   | DPCZ     |
| 399373.XSHE | 大盘价值                       | 2010-01-04 | 2010-01-04   | DPJZ     |
| 399374.XSHE | 中盘成长                       | 2010-01-04 | 2010-01-04   | ZPCZ     |
| 399375.XSHE | 中盘价值                       | 2010-01-04 | 2010-01-04   | ZPJZ     |
| 399376.XSHE | 小盘成长                       | 2010-01-04 | 2010-01-04   | XPCZ     |
| 399377.XSHE | 小盘价值                       | 2010-01-04 | 2010-01-04   | XPJZ     |
| 399378.XSHE | 南方低碳                       | 2010-09-20 | 2010-09-20   | NFDT     |
| 399379.XSHE | 国证基金                       | 2011-12-02 | 2011-12-02   | GZJJ     |
| 399380.XSHE | 国证ETF                        | 2011-12-02 | 2011-12-02   | GZETF    |
| 399381.XSHE | 1000能源                       | 2011-12-02 | 2011-12-02   | 1000NY   |
| 399382.XSHE | 1000材料                       | 2011-12-02 | 2011-12-02   | 1000CL   |
| 399383.XSHE | 1000工业                       | 2011-12-02 | 2011-12-02   | 1000GY   |
| 399384.XSHE | 1000可选                       | 2011-12-02 | 2011-12-02   | 1000KX   |
| 399385.XSHE | 1000消费                       | 2011-12-02 | 2011-12-02   | 1000XF   |
| 399386.XSHE | 1000医药                       | 2011-12-02 | 2011-12-02   | 1000YY   |
| 399387.XSHE | 1000金融                       | 2011-12-02 | 2011-12-02   | 1000JR   |
| 399388.XSHE | 1000信息                       | 2011-12-02 | 2011-12-02   | 1000XX   |
| 399389.XSHE | 国证通信                       | 2011-12-02 | 2011-12-02   | GZTX     |
| 399390.XSHE | 1000公用                       | 2011-12-02 | 2011-12-02   | 1000GY   |
| 399391.XSHE | 投资时钟                       | 2012-03-26 | 2012-03-26   | TZSZ     |
| 399392.XSHE | 国证新兴                       | 2012-03-28 | 2012-03-28   | GZXX     |
| 399393.XSHE | 国证地产                       | 2012-08-20 | 2012-08-20   | GZDC     |
| 399394.XSHE | 国证医药                       | 2012-10-29 | 2012-10-29   | GZYY     |
| 399395.XSHE | 国证有色                       | 2012-10-29 | 2012-10-29   | GZYS     |
| 399396.XSHE | 国证食品                       | 2012-10-29 | 2012-10-29   | GZSP     |
| 399397.XSHE | OCT文化                        | 2012-11-09 | 2012-11-09   | OCTWH    |
| 399398.XSHE | 绩效指数                       | 2012-11-19 | 2012-11-19   | JXZS     |
| 399399.XSHE | 中经GDP                        | 2012-11-23 | 2012-11-23   | ZJGDP    |
| 399400.XSHE | 大中盘                         | 2013-03-20 | 2013-03-20   | DZP      |
| 399401.XSHE | 中小盘                         | 2013-03-20 | 2013-03-20   | ZXP      |
| 399402.XSHE | 周期100                        | 2013-10-28 | 2013-10-28   | ZQ100    |
| 399403.XSHE | 防御100                        | 2013-10-28 | 2013-10-28   | FY100    |
| 399404.XSHE | 大盘低波                       | 2013-12-05 | 2013-12-05   | DPDB     |
| 399405.XSHE | 大盘高贝                       | 2013-12-05 | 2013-12-05   | DPGB     |
| 399406.XSHE | 中盘低波                       | 2013-12-05 | 2013-12-05   | ZPDB     |
| 399407.XSHE | 中盘高贝                       | 2013-12-05 | 2013-12-05   | ZPGB     |
| 399408.XSHE | 小盘低波                       | 2013-12-05 | 2013-12-05   | XPDB     |
| 399409.XSHE | 小盘高贝                       | 2013-12-05 | 2013-12-05   | XPGB     |
| 399410.XSHE | 苏州率先                       | 2013-12-12 | 2013-12-12   | SZLX     |
| 399411.XSHE | 红利100                        | 2014-04-10 | 2014-04-10   | HL100    |
| 399412.XSHE | 国证新能                       | 2014-05-16 | 2014-05-16   | GZXN     |
| 399413.XSHE | 国证转债                       | 2014-08-27 | 2014-08-27   | GZZZ     |
| 399415.XSHE | I100                           | 2014-09-12 | 2014-09-12   | I100     |
| 399416.XSHE | I300                           | 2014-09-12 | 2014-09-12   | I300     |
| 399417.XSHE | 国证新能源汽车指数             | 2014-09-24 | 2014-09-24   | XNYC     |
| 399418.XSHE | 国证国家安全指数               | 2014-11-21 | 2014-11-21   | GZGA     |
| 399419.XSHE | 国证高铁指数                   | 2014-12-10 | 2014-12-10   | GZGT     |
| 399420.XSHE | 国证保险证券指数               | 2014-12-10 | 2014-12-10   | GZBZ     |
| 399423.XSHE | 中关村50指数                   | 2015-02-05 | 2015-02-05   | ZGC50    |
| 399427.XSHE | 国证德高行专利领先指数         | 2015-02-17 | 2015-02-17   | ZLLX     |
| 399428.XSHE | 国证定向增发指数               | 2015-02-17 | 2015-02-17   | GZDZ     |
| 399429.XSHE | 新丝路指数                     | 2015-01-08 | 2015-01-08   | XSL      |
| 399431.XSHE | 国证银行行业指数               | 2014-12-30 | 2014-12-30   | GZYH     |
| 399432.XSHE | 国证汽车与汽车零配件行业指数   | 2014-12-30 | 2014-12-30   | GZQC     |
| 399433.XSHE | 国证交通运输行业指数           | 2014-12-30 | 2014-12-30   | GZJY     |
| 399434.XSHE | 国证传媒行业指数               | 2014-12-30 | 2014-12-30   | GZCM     |
| 399435.XSHE | 国证农牧渔产品行业指数         | 2014-12-30 | 2014-12-30   | GZNM     |
| 399436.XSHE | 国证煤炭行业指数               | 2014-12-30 | 2014-12-30   | GZMT     |
| 399437.XSHE | 国证证券行业指数               | 2014-12-30 | 2014-12-30   | GZZQ     |
| 399438.XSHE | 国证电力公用事业行业指数       | 2014-12-30 | 2014-12-30   | GZDL     |
| 399439.XSHE | 国证石油天然气行业指数         | 2014-12-30 | 2014-12-30   | GZYQ     |
| 399440.XSHE | 国证黑色金属行业指数           | 2014-12-30 | 2014-12-30   | GZGT     |
| 399441.XSHE | 国证生物医药指数               | 2015-01-20 | 2015-01-20   | SWYY     |
| 399481.XSHE | 企债指数                       | 2003-01-02 | 2005-01-04   | QZZS     |
| 399550.XSHE | 央视50                         | 2012-06-06 | 2012-06-06   | YS50     |
| 399551.XSHE | 央视创新                       | 2013-06-06 | 2013-06-06   | YSCX     |
| 399552.XSHE | 央视成长                       | 2013-06-06 | 2013-06-06   | YSCZ     |
| 399553.XSHE | 央视回报                       | 2013-06-06 | 2013-06-06   | YSHB     |
| 399554.XSHE | 央视治理                       | 2013-06-06 | 2013-06-06   | YSZL     |
| 399555.XSHE | 央视责任                       | 2013-06-06 | 2013-06-06   | YSZR     |
| 399556.XSHE | 央视生态                       | 2014-06-06 | 2014-06-06   | YSST     |
| 399557.XSHE | 央视文化                       | 2014-06-06 | 2014-06-06   | YSWH     |
| 399602.XSHE | 中小成长                       | 2010-05-24 | 2010-05-24   | ZXCZ     |
| 399604.XSHE | 中小价值                       | 2010-05-24 | 2010-05-24   | ZXJZ     |
| 399606.XSHE | 创业板R                        | 2010-06-01 | 2010-06-01   | CYBR     |
| 399608.XSHE | 科技100                        | 2010-10-18 | 2010-10-18   | KJ100    |
| 399610.XSHE | TMT50                          | 2010-11-08 | 2010-11-08   | TMT50    |
| 399611.XSHE | 中创100R                       | 2011-02-28 | 2011-02-28   | ZC100R   |
| 399612.XSHE | 中创100                        | 2011-02-28 | 2011-02-28   | ZC100    |
| 399613.XSHE | 深证能源                       | 2011-06-15 | 2011-06-15   | SZNY     |
| 399614.XSHE | 深证材料                       | 2011-06-15 | 2011-06-15   | SZCL     |
| 399615.XSHE | 深证工业                       | 2011-06-15 | 2011-06-15   | SZGY     |
| 399616.XSHE | 深证可选                       | 2011-06-15 | 2011-06-15   | SZKX     |
| 399617.XSHE | 深证消费                       | 2011-06-15 | 2011-06-15   | SZXF     |
| 399618.XSHE | 深证医药                       | 2011-06-15 | 2011-06-15   | SZYY     |
| 399619.XSHE | 深证金融                       | 2011-06-15 | 2011-06-15   | SZJR     |
| 399620.XSHE | 深证信息                       | 2011-06-15 | 2011-06-15   | SZXX     |
| 399621.XSHE | 深证电信                       | 2011-06-15 | 2011-06-15   | SZDX     |
| 399622.XSHE | 深证公用                       | 2011-06-15 | 2011-06-15   | SZGY     |
| 399623.XSHE | 中小基础                       | 2011-07-25 | 2011-07-25   | ZXJC     |
| 399624.XSHE | 中创400                        | 2011-08-15 | 2011-08-15   | ZC400    |
| 399625.XSHE | 中创500                        | 2011-08-15 | 2011-08-15   | ZC500    |
| 399626.XSHE | 中创成长                       | 2011-08-15 | 2011-08-15   | ZCCZ     |
| 399627.XSHE | 中创价值                       | 2011-08-15 | 2011-08-15   | ZCJZ     |
| 399628.XSHE | 700成长                        | 2011-09-01 | 2011-09-01   | 700CZ    |
| 399629.XSHE | 700价值                        | 2011-09-01 | 2011-09-01   | 700JZ    |
| 399630.XSHE | 1000成长                       | 2011-09-01 | 2011-09-01   | 1000CZ   |
| 399631.XSHE | 1000价值                       | 2011-09-01 | 2011-09-01   | 1000JZ   |
| 399632.XSHE | 深100EW                        | 2011-10-28 | 2011-10-28   | S100EW   |
| 399633.XSHE | 深300EW                        | 2011-10-28 | 2011-10-28   | S300EW   |
| 399634.XSHE | 中小板EW                       | 2011-10-28 | 2011-10-28   | ZXBEW    |
| 399635.XSHE | 创业板EW                       | 2011-10-28 | 2011-10-28   | CYBEW    |
| 399636.XSHE | 深证装备                       | 2011-11-15 | 2011-11-15   | SZZB     |
| 399637.XSHE | 深证地产                       | 2011-11-15 | 2011-11-15   | SZDC     |
| 399638.XSHE | 深证环保                       | 2011-11-15 | 2011-11-15   | SZHB     |
| 399639.XSHE | 深证大宗                       | 2011-11-15 | 2011-11-15   | SZDZ     |
| 399640.XSHE | 创业基础                       | 2012-01-16 | 2012-01-16   | CYJC     |
| 399641.XSHE | 深证新兴                       | 2012-02-01 | 2012-02-01   | SZXX     |
| 399642.XSHE | 中小新兴                       | 2012-02-01 | 2012-02-01   | ZXXX     |
| 399643.XSHE | 创业新兴                       | 2012-02-01 | 2012-02-01   | CYXX     |
| 399644.XSHE | 深证时钟                       | 2012-03-26 | 2012-03-26   | SZSZ     |
| 399645.XSHE | 100低波                        | 2012-06-12 | 2012-06-12   | 100DB    |
| 399646.XSHE | 深消费50                       | 2012-08-06 | 2012-08-06   | SXF50    |
| 399647.XSHE | 深医药50                       | 2012-08-06 | 2012-08-06   | SYY50    |
| 399648.XSHE | 深证GDP                        | 2012-08-08 | 2012-08-08   | SZGDP    |
| 399649.XSHE | 中小红利                       | 2012-08-20 | 2012-08-20   | ZXHL     |
| 399650.XSHE | 中小治理                       | 2012-08-20 | 2012-08-20   | ZXZL     |
| 399651.XSHE | 中小责任                       | 2012-08-20 | 2012-08-20   | ZXZR     |
| 399652.XSHE | 中创高新                       | 2012-09-17 | 2012-09-17   | ZCGX     |
| 399653.XSHE | 深证龙头                       | 2012-09-25 | 2012-09-25   | SZLT     |
| 399654.XSHE | 深证文化                       | 2012-11-09 | 2012-11-09   | SZWH     |
| 399655.XSHE | 深证绩效                       | 2012-11-19 | 2012-11-19   | SZJX     |
| 399656.XSHE | 100绩效                        | 2012-11-19 | 2012-11-19   | 100JX    |
| 399657.XSHE | 300绩效                        | 2012-11-19 | 2012-11-19   | 300JX    |
| 399658.XSHE | 中小绩效                       | 2012-11-19 | 2012-11-19   | ZXJX     |
| 399659.XSHE | 深成指EW                       | 2012-11-23 | 2012-11-23   | SCZEW    |
| 399660.XSHE | 中创EW                         | 2012-11-23 | 2012-11-23   | ZCEW     |
| 399661.XSHE | 深证低波                       | 2012-12-20 | 2012-12-20   | SZDB     |
| 399662.XSHE | 深证高贝                       | 2012-12-20 | 2012-12-20   | SZGB     |
| 399663.XSHE | 中小低波                       | 2012-12-20 | 2012-12-20   | ZXDB     |
| 399664.XSHE | 中小高贝                       | 2012-12-20 | 2012-12-20   | ZXGB     |
| 399665.XSHE | 中创低波                       | 2012-12-20 | 2012-12-20   | ZCDB     |
| 399666.XSHE | 中创高贝                       | 2012-12-20 | 2012-12-20   | ZCGB     |
| 399667.XSHE | 创业板G                        | 2013-01-07 | 2013-01-07   | CYBG     |
| 399668.XSHE | 创业板V                        | 2013-01-07 | 2013-01-07   | CYBV     |
| 399669.XSHE | 深证农业                       | 2013-06-24 | 2013-06-24   | SZNY     |
| 399670.XSHE | 深周期50                       | 2013-10-28 | 2013-10-28   | SZQ50    |
| 399671.XSHE | 深防御50                       | 2013-10-28 | 2013-10-28   | SFY50    |
| 399672.XSHE | 深红利50                       | 2014-04-10 | 2014-04-10   | SHL50    |
| 399673.XSHE | 创业板50                       | 2014-06-18 | 2014-06-18   | CYB50    |
| 399674.XSHE | 深A医药卫生指数                | 2015-06-08 | 2015-06-08   | SAYY     |
| 399675.XSHE | 深A软件与互联网指数            | 2015-06-08 | 2015-06-08   | SHLW     |
| 399676.XSHE | 深A医药卫生等权指数            | 2015-06-08 | 2015-06-08   | SYYEW    |
| 399677.XSHE | 深A软件与互联网等权指数        | 2015-06-08 | 2015-06-08   | SHLEW    |
| 399678.XSHE | 深证次新股指数                 | 2015-06-18 | 2015-06-18   | SCXG     |
| 399679.XSHE | 深证200指数                    | 2015-06-18 | 2015-06-18   | SZ200    |
| 399680.XSHE | 深成能源行业指数               | 2015-08-31 | 2015-08-31   | SCNY     |
| 399681.XSHE | 深成原材料行业指数             | 2015-08-31 | 2015-08-31   | SCCL     |
| 399682.XSHE | 深成工业行业指数               | 2015-08-31 | 2015-08-31   | SCGY     |
| 399683.XSHE | 深成可选消费行业指数           | 2015-08-31 | 2015-08-31   | SCKX     |
| 399684.XSHE | 深成主要消费行业指数           | 2015-08-31 | 2015-08-31   | SCXF     |
| 399685.XSHE | 深成医药卫生行业指数           | 2015-08-31 | 2015-08-31   | SCYY     |
| 399686.XSHE | 深成金融地产行业指数           | 2015-08-31 | 2015-08-31   | SCJR     |
| 399687.XSHE | 深成信息技术行业指数           | 2015-08-31 | 2015-08-31   | SCXX     |
| 399688.XSHE | 深成电信业务行业指数           | 2015-08-31 | 2015-08-31   | SCDX     |
| 399689.XSHE | 深成公用事业行业指数           | 2015-08-31 | 2015-08-31   | SCGY     |
| 399690.XSHE | 深证中小板专利领先指数         | 2016-05-23 | 2016-05-23   | ZXZL     |
| 399691.XSHE | 深证创业板专利领先指数         | 2016-05-23 | 2016-05-23   | CYZL     |
| 399692.XSHE | 创业300低波动率指数            | 2016-12-20 | 2016-12-20   | CYDB     |
| 399693.XSHE | 安防产业                       | 2016-06-20 | 2016-06-20   | AFCY     |
| 399694.XSHE | 创业高贝                       | 2016-12-20 | 2016-12-20   | CYGB     |
| 399695.XSHE | 深证节能环保指数               | 2017-04-28 | 2017-04-28   | SZJN     |
| 399696.XSHE | 深证创投                       | 2016-10-21 | 2016-10-21   | SZCT     |
| 399697.XSHE | 中关村60                       | 2017-01-10 | 2017-01-10   | ZGC60    |
| 399698.XSHE | 深证优势成长                   | 2017-06-15 | 2017-06-15   | YSCZ     |
| 399699.XSHE | 金融科技                       | 2017-06-09 | 2017-06-09   | JRKJ     |
| 399701.XSHE | 深证F60                        | 2010-05-10 | 2010-05-10   | SZF60    |
| 399702.XSHE | 深证F120                       | 2010-05-10 | 2010-05-10   | SZF120   |
| 399703.XSHE | 深证F200                       | 2010-05-10 | 2010-05-10   | SZF200   |
| 399704.XSHE | 深证上游                       | 2011-10-18 | 2011-10-18   | SZSY     |
| 399705.XSHE | 深证中游                       | 2011-10-18 | 2011-10-18   | SZZY     |
| 399706.XSHE | 深证下游                       | 2011-10-18 | 2011-10-18   | SZXY     |
| 399707.XSHE | 中证申万证券行业指数           | 2015-04-16 | 2015-04-16   | CSSWZQ   |
| 399802.XSHE | 500深市                        | 2012-09-05 | 2012-09-05   | 500SS    |
| 399803.XSHE | 中证工业4.0指数                | 2015-03-10 | 2015-03-10   | GY4.0    |
| 399804.XSHE | 中证体育产业指数               | 2015-02-09 | 2015-02-09   | ZZTY     |
| 399805.XSHE | 中证互联网金融指数             | 2015-02-10 | 2015-02-10   | HLJR     |
| 399806.XSHE | 中证环境治理指数               | 2014-07-21 | 2014-07-21   | HJZL     |
| 399807.XSHE | 中证高铁产业指数               | 2015-01-20 | 2015-01-20   | GTCY     |
| 399808.XSHE | 中证新能源指数                 | 2015-02-10 | 2015-02-10   | ZZXN     |
| 399809.XSHE | 中证方正富邦保险主题指数       | 2015-02-13 | 2015-02-13   | BXZT     |
| 399810.XSHE | 中证申万传媒行业投资指数       | 2015-08-03 | 2015-08-03   | CSSWCM   |
| 399811.XSHE | 中证申万电子行业投资指数       | 2015-08-03 | 2015-08-03   | CSSWDZ   |
| 399812.XSHE | 中证养老产业指数               | 2014-06-06 | 2014-06-06   | YLCY     |
| 399813.XSHE | 中证国防安全指数               | 2015-01-29 | 2015-01-29   | ZZGA     |
| 399814.XSHE | 中证大农业指数                 | 2014-07-22 | 2014-07-22   | DNY      |
| 399817.XSHE | 中证阿拉善生态主题100指数      | 2015-10-21 | 2015-10-21   | ST100    |
| 399901.XSHE | 中证南方小康产业指数           | 2010-04-01 | 2010-04-01   | XKZS     |
| 399902.XSHE | 中证流通指数                   | 2006-02-27 | 2006-02-27   | ZZLT     |
| 399903.XSHE | 中证100指数                    | 2006-05-29 | 2006-05-29   | ZZ100    |
| 399904.XSHE | 中证中盘200指数                | 2007-01-15 | 2007-01-15   | ZZ200    |
| 399905.XSHE | 中证500                        | 2007-01-15 | 2007-01-15   | ZZ500    |
| 399906.XSHE | 中证800指数                    | 2007-01-15 | 2007-01-15   | ZZ800    |
| 399907.XSHE | 中证中小盘700指数              | 2007-01-15 | 2007-01-15   | ZZ700    |
| 399908.XSHE | 沪深300能源指数                | 2007-07-02 | 2007-07-02   | 300NY    |
| 399909.XSHE | 沪深300原材料指数              | 2007-07-02 | 2007-07-02   | 300CL    |
| 399910.XSHE | 沪深300工业指数                | 2007-07-02 | 2007-07-02   | 300GY    |
| 399911.XSHE | 沪深300可选消费指数            | 2007-07-02 | 2007-07-02   | 300KX    |
| 399912.XSHE | 沪深300主要消费指数            | 2007-07-02 | 2007-07-02   | 300XF    |
| 399913.XSHE | 沪深300医药卫生指数            | 2007-07-02 | 2007-07-02   | 300YY    |
| 399914.XSHE | 沪深300金融地产指数            | 2007-07-02 | 2007-07-02   | 300JR    |
| 399915.XSHE | 沪深300信息技术指数            | 2007-07-02 | 2007-07-02   | 300XX    |
| 399916.XSHE | 沪深300电信业务指数            | 2007-07-02 | 2007-07-02   | 300DX    |
| 399917.XSHE | 沪深300公用事业指数            | 2007-07-02 | 2007-07-02   | 300GY    |
| 399918.XSHE | 沪深300成长指数                | 2008-01-21 | 2008-01-21   | 300CZ    |
| 399919.XSHE | 沪深300价值指数                | 2008-01-21 | 2008-01-21   | 300JZ    |
| 399920.XSHE | 沪深300相对成长指数            | 2008-01-21 | 2008-01-21   | 300RCZ   |
| 399922.XSHE | 中证红利指数                   | 2008-05-26 | 2008-05-26   | ZZHL     |
| 399923.XSHE | 公司债                         | 2008-11-19 | 2008-11-19   | GSZ      |
| 399925.XSHE | 中证锐联基本面50指数           | 2009-02-26 | 2009-02-26   | JBM50    |
| 399926.XSHE | 中证中央企业综合指数           | 2009-03-30 | 2009-03-30   | ZZYQ     |
| 399927.XSHE | 中证中央企业100指数            | 2009-03-30 | 2009-03-30   | YQ100    |
| 399928.XSHE | 中证能源指数                   | 2009-07-03 | 2009-07-03   | ZZNY     |
| 399929.XSHE | 中证原材料指数                 | 2009-07-03 | 2009-07-03   | ZZCL     |
| 399930.XSHE | 中证工业指数                   | 2009-07-03 | 2009-07-03   | ZZGY     |
| 399931.XSHE | 中证可选消费指数               | 2009-07-03 | 2009-07-03   | ZZKX     |
| 399932.XSHE | 中证主要消费指数               | 2009-07-03 | 2009-07-03   | ZZXF     |
| 399933.XSHE | 中证医药卫生指数               | 2009-07-03 | 2009-07-03   | ZZYY     |
| 399934.XSHE | 中证金融地产指数               | 2009-07-03 | 2009-07-03   | ZZJR     |
| 399935.XSHE | 中证信息技术指数               | 2009-07-03 | 2009-07-03   | ZZXX     |
| 399936.XSHE | 中证电信业务指数               | 2009-07-03 | 2009-07-03   | ZZDX     |
| 399937.XSHE | 中证公用事业指数               | 2009-07-03 | 2009-07-03   | ZZGY     |
| 399938.XSHE | 中证民营企业综合指数           | 2009-08-25 | 2009-08-25   | ZZMQ     |
| 399939.XSHE | 中证民营企业200指数            | 2009-08-25 | 2009-08-25   | MQ200    |
| 399940.XSHE | 中证财富大盘指数               | 2009-09-25 | 2009-09-25   | CFDP     |
| 399941.XSHE | 中证内地新能源主题指数         | 2009-10-28 | 2009-10-28   | XNY      |
| 399942.XSHE | 中证内地消费主题指数           | 2009-10-28 | 2009-10-28   | NDXF     |
| 399943.XSHE | 中证内地基建主题指数           | 2009-10-28 | 2009-10-28   | NDJJ     |
| 399944.XSHE | 中证内地资源主题指数           | 2009-10-28 | 2009-10-28   | NDZY     |
| 399945.XSHE | 中证内地运输主题指数           | 2009-10-28 | 2009-10-28   | NDYS     |
| 399946.XSHE | 中证内地金融主题指数           | 2009-10-28 | 2009-10-28   | NDJR     |
| 399947.XSHE | 中证内地银行主题指数           | 2009-10-28 | 2009-10-28   | NDYH     |
| 399948.XSHE | 中证内地地产主题指数           | 2009-10-28 | 2009-10-28   | NDDC     |
| 399949.XSHE | 中证内地农业主题指数           | 2009-10-28 | 2009-10-28   | NDNY     |
| 399950.XSHE | 沪深300基建主题指数            | 2009-10-28 | 2009-10-28   | 300JJ    |
| 399951.XSHE | 沪深300银行指数                | 2009-10-28 | 2009-10-28   | 300YH    |
| 399952.XSHE | 沪深300地产指数                | 2009-10-28 | 2009-10-28   | 300DC    |
| 399953.XSHE | 中证地方国有企业综合指数       | 2010-02-09 | 2010-02-09   | ZZDQ     |
| 399954.XSHE | 中证地方国有企业100指数        | 2010-02-09 | 2010-02-09   | DQ100    |
| 399955.XSHE | 中证国有企业综合指数           | 2010-02-09 | 2010-02-09   | ZZGQ     |
| 399956.XSHE | 中证国有企业200指数            | 2010-02-09 | 2010-02-09   | GQ200    |
| 399957.XSHE | 沪深300运输指数                | 2009-06-16 | 2009-06-16   | 300YS    |
| 399958.XSHE | 中证创业成长指数               | 2010-03-24 | 2010-03-24   | CYCZ     |
| 399959.XSHE | 军工指数                       | 2011-08-30 | 2011-08-30   | JGZS     |
| 399960.XSHE | 中证龙头企业指数               | 2010-02-09 | 2010-02-09   | ZZLT     |
| 399961.XSHE | 中证上游资源产业指数           | 2010-04-16 | 2010-04-16   | ZZSY     |
| 399962.XSHE | 中证中游制造产业指数           | 2010-04-16 | 2010-04-16   | ZZZY     |
| 399963.XSHE | 中证下游消费与服务产业指数     | 2010-04-16 | 2010-04-16   | ZZXY     |
| 399964.XSHE | 中证新兴产业指数               | 2010-04-30 | 2010-04-30   | ZZXX     |
| 399965.XSHE | 800地产                        | 2014-04-04 | 2014-04-04   | 800DC    |
| 399966.XSHE | 800非银                        | 2014-04-04 | 2014-04-04   | 800FY    |
| 399967.XSHE | 中证军工                       | 2013-12-26 | 2013-12-26   | ZZJG     |
| 399968.XSHE | 沪深300周期行业指数            | 2010-05-28 | 2010-05-28   | 300ZQ    |
| 399969.XSHE | 沪深300非周期行业指数          | 2010-05-28 | 2010-05-28   | 300FZ    |
| 399970.XSHE | 中证移动互联网指数             | 2014-05-05 | 2014-05-05   | YDHL     |
| 399971.XSHE | 中证传媒指数                   | 2014-04-15 | 2014-04-15   | ZZCM     |
| 399972.XSHE | 300深市                        | 2012-06-21 | 2012-06-21   | 300SS    |
| 399973.XSHE | 中证国防指数                   | 2014-04-15 | 2014-04-15   | ZZGF     |
| 399974.XSHE | 中证国有企业改革指数           | 2014-08-07 | 2014-08-07   | GQGG     |
| 399975.XSHE | 中证全指证券公司指数(四级行业) | 2013-07-15 | 2013-07-15   | ZQGS     |
| 399976.XSHE | 中证新能源汽车指数             | 2014-11-28 | 2014-11-28   | XNYC     |
| 399977.XSHE | 中证内地低碳经济主题指数       | 2011-01-21 | 2011-01-21   | NDDT     |
| 399978.XSHE | 中证医药100指数                | 2011-03-18 | 2011-03-18   | YY100    |
| 399979.XSHE | 中证大宗商品股票指数           | 2011-08-22 | 2011-08-22   | DZSP     |
| 399980.XSHE | 中证超级大盘指数               | 2011-05-10 | 2011-05-10   | ZZCD     |
| 399981.XSHE | 沪深300行业分层等权重指数      | 2011-06-13 | 2011-06-13   | 300FC    |
| 399982.XSHE | 中证500等权重指数              | 2011-06-13 | 2011-06-13   | 500DQ    |
| 399983.XSHE | 沪深300地产等权重指数          | 2013-11-22 | 2013-11-22   | DCDQ     |
| 399984.XSHE | 沪深300等权重指数              | 2011-08-02 | 2011-08-02   | 300DQ    |
| 399985.XSHE | 中证全指指数                   | 2011-08-02 | 2011-08-02   | ZZQZ     |
| 399986.XSHE | 中证银行指数                   | 2013-07-15 | 2013-07-15   | ZZYH     |
| 399987.XSHE | 中证酒指数                     | 2014-12-10 | 2014-12-10   | ZZJ      |
| 399989.XSHE | 中证医疗指数                   | 2014-10-31 | 2014-10-31   | ZZYL     |
| 399990.XSHE | 中证煤炭等权指数               | 2015-01-21 | 2015-01-21   | MTDQ     |
| 399991.XSHE | 中证一带一路主题指数           | 2015-02-16 | 2015-02-16   | YDYL     |
| 399992.XSHE | 中证万得并购重组指数           | 2015-05-08 | 2015-05-08   | CSWDBG   |
| 399993.XSHE | 中证万得生物科技指数           | 2015-05-08 | 2015-05-08   | CSWDSK   |
| 399994.XSHE | 中证信息安全主题指数           | 2015-03-12 | 2015-03-12   | XXAQ     |
| 399995.XSHE | 中证基建工程指数               | 2015-03-12 | 2015-03-12   | JJGC     |
| 399996.XSHE | 中证智能家居指数               | 2014-09-17 | 2014-09-17   | ZNJJ     |
| 399997.XSHE | 中证白酒指数                   | 2015-01-21 | 2015-01-21   | ZZBJ     |
| 399998.XSHE | 中证煤炭指数                   | 2015-02-13 | 2015-02-13   | ZZMT     |

 

## 获取国际指数行情数据

```
from jqdatasdk import *
finance.run_query(query(finance.GLOBAL_IDX_DAILY).filter(finance.GLOBAL_IDX_DAILY.code==code).limit(n))
```

获取港美股指数日行情数据

**参数**

- **query(finance.GLOBAL_IDX_DAILY)**：表示从finance.GLOBAL_IDX_DAILY这张表中查询港美股指数日行情的数据，还可以指定所要查询的字段名，格式如下：query(库名.表名.字段名1，库名.表名.字段名2），多个字段用逗号分隔进行提取；query函数的更多用法详见：[sqlalchemy.orm.query.Query对象](http://docs.sqlalchemy.org/en/rel_1_0/orm/query.html)
- **finance.GLOBAL_IDX_DAILY**：收录了港美股各个指数数据，表结构和字段信息如下：

**字段设计**

| 字段       | 名称     | 类型          | 释义                                   |
| ---------- | -------- | ------------- | -------------------------------------- |
| code       | 指数代码 | varchar(64)   | 代码列表详见下方国际指数代码名称对照表 |
| name       | 指数名称 | varchar(64)   |                                        |
| day        | 日期     | date          |                                        |
| open       | 开盘价   | decimal(20,6) |                                        |
| close      | 收盘价   | decimal(20,6) |                                        |
| low        | 最低价   | decimal(20,6) |                                        |
| high       | 最高价   | decimal(20,6) |                                        |
| volume     | 成交量   | decimal(20,6) |                                        |
| change_pct | 涨跌幅   | decimal(20,4) |                                        |
| pre_close  | 前收价   | decimal(20,6) |                                        |

**国际指数代码名称对照表**

| 指数代码    | 指数名称             |
| ----------- | -------------------- |
| 800151.XHKG | 红筹指数             |
| 800100.XHKG | 国企指数             |
| 800000.XHKG | 恒生指数             |
| INX         | 标普500指数          |
| IXIC        | 纳斯达克综合指数     |
| DJI         | 道琼斯指数           |
| KS11        | 韩国综合指数         |
| FTSE        | 英国富时             |
| RTS         | 俄罗斯RTS指数        |
| MIB         | 意大利MIB指数        |
| GDAXI       | 德国法兰克福DAX指数  |
| N225        | 东京日经225指数      |
| IBEX        | 西班牙IBEX35         |
| FCHI        | 法国巴黎CAC40指数    |
| IBOV        | 圣保罗IBOVESPA指数   |
| MXX         | 墨西哥MXX指数        |
| GSPTSE      | 多伦多股票交易所综合 |

- **filter(finance.GLOBAL_IDX_DAILY.code==code)**：指定筛选条件，通过finance.GLOBAL_IDX_DAILY.code==code可以指定您想要查询的指数代码；除此之外，还可以对表中其他字段指定筛选条件，如finance.GLOBAL_IDX_DAILY.day.desc()，表示按天倒序排名。
- **limit(n)**：限制返回的数据条数，n指定返回条数。

**返回结果：**

- 返回一个 dataframe，每一行对应数据表中的一条数据， 列索引是您所查询的字段名称

**注意：**

1. **为了防止返回数据量过大, 我们每次最多返回3000行**
2. 不能进行连表查询，即同时查询多张表的数据

**示例：**

**获取港美股指数日行情数据示例：**

In [9]:
#查询美股指数标普500('INX')近十天的指数数据
q=query(finance.GLOBAL_IDX_DAILY).filter(finance.GLOBAL_IDX_DAILY.code=='INX').order_by(finance.GLOBAL_IDX_DAILY.day.desc()).limit(10)
df=finance.run_query(q)
df.head()

,id,code,name,day,open,close,low,high,volume,change_pct,pre_close
0,187253,INX,标普500指数,2019-07-24,2998.77,3019.56,2996.82,3019.59,1.925227e+09,0.004688,3005.47
1,187225,INX,标普500指数,2019-07-23,2994.74,3005.47,2988.56,3005.90,1.870689e+09,0.006848,2985.03
2,187197,INX,标普500指数,2019-07-22,2981.93,2985.03,2976.65,2990.71,1.667005e+09,0.002829,2976.61
3,187172,INX,标普500指数,2019-07-19,3004.26,2976.61,2975.86,3006.02,1.872665e+09,-0.006177,2995.11
4,187145,INX,标普500指数,2019-07-18,2978.87,2995.11,2973.09,2998.28,1.793633e+09,0.003582,2984.42
